In [127]:
from rapidfuzz import process, fuzz

# Danh sách predefined
list_xa = ["phù ninh", "liên ninh", "hoa binh"]
list_huyen = ["phu ninh", "vinh tuong", "ba vi"]
list_tinh = ["phu tho", "vinh phuc", "ha noi"]

# Text ASR tự do, có thể xáo trộn / sai chính tả
text_asr = "tinh phu tho xa phu nin huyen phu nin"

# Chuẩn hóa
text_norm = text_asr.lower().replace(",", "").replace(".", "").strip()
tokens = text_norm.split()

# Sinh n-gram 1-3 từ
ngrams = []
for i in range(len(tokens)):
    for j in range(1, 4):  # n=1..3
        if i+j <= len(tokens):
            ngrams.append(" ".join(tokens[i:i+j]))

# Hàm fuzzy match với rapidfuzz
def match_best_ngram(ngrams, reference_list, threshold=80):
    best_score = 0
    best_match = None
    for ng in ngrams:
        match = process.extractOne(
            ng, reference_list, scorer=fuzz.token_sort_ratio
        )
        if match:
            candidate, score, _ = match
            if score > best_score and score >= threshold:
                best_score = score
                best_match = candidate
    return best_match

# Match từng cấp
tinh = match_best_ngram(ngrams, list_tinh)
huyen = match_best_ngram(ngrams, list_huyen)
xa = match_best_ngram(ngrams, list_xa)

print("Xã:", xa)
print("Huyện:", huyen)
print("Tỉnh:", tinh)


Xã: phù ninh
Huyện: phu ninh
Tỉnh: phu tho


In [123]:
# dùng full-address list (district-level) để giúp disambiguation / match chính xác hơn
# file format (tsv/csv): STT\tTên\tCấp\tTỉnh / Thành Phố
# ví dụ dòng: 1\tQuận Ba Đình\tQuận\tThành phố Hà Nội

from rapidfuzz import fuzz, process
from collections import defaultdict
import re

# -------------------------------
# Utility (reuse)
# -------------------------------
def remove_prefix(name):
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name, flags=re.I)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name, flags=re.I)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name, flags=re.I)
    return name.strip()

def normalize(s):
    return remove_prefix(s).lower().strip()

def match_fuzzy_topk(span, candidate_list, top_k=5, scorer=fuzz.WRatio):
    if not candidate_list:
        return []
    return [(r[0], r[1]) for r in process.extract(span, candidate_list, scorer=scorer, limit=top_k)]

# -------------------------------
# Load full-address list (district -> province), build full_address strings
# -------------------------------
def load_full_districts(file_path, delimiter="\t"):
    """
    Return:
      - full_addresses: list of strings "Quận Ba Đình, Thành phố Hà Nội"
      - district_to_provinces: dict district_name -> [province_name(s)]
      - full_map: dict full_address_string -> (district, province, level)
    """
    full_addresses = []
    district_to_provinces = defaultdict(set)
    full_map = {}
    with open(file_path, "r", encoding="utf-8") as f:
        header = next(f)
        for line in f:
            if not line.strip():
                continue
            parts = [p.strip() for p in line.strip().split(delimiter)]
            # attempt tolerant parsing
            if len(parts) < 4:
                continue
            _, name, level, province = parts[:4]
            full_str = f"{name}, {province}"
            full_addresses.append(full_str)
            district_to_provinces[name].add(province)
            full_map[full_str] = (name, province, level)
    district_to_provinces = {k: list(v) for k, v in district_to_provinces.items()}
    return full_addresses, district_to_provinces, full_map

# -------------------------------
# Generate candidates INCLUDING full-address matches
# -------------------------------
def gen_candidates_with_full(span, wards_list, districts_list, provinces_list,
                             ward_to_districts, district_to_provinces,
                             normalized_ward_map, normalized_district_map, normalized_province_map,
                             full_addresses, full_map,
                             top_k=6):
    span_norm = normalize(span)
    candidates = []

    # 1) try match full-address strings (strong signal)
    top_full = match_fuzzy_topk(span, full_addresses, top_k=top_k)
    for full_name, score in top_full:
        district, province, level = full_map[full_name]
        # treat as district-candidate but with context (full)
        candidates.append({
            "type": "full_address",
            "name": full_name,
            "score": score + 10,  # boost full match a bit
            "details": {"district": district, "province": province, "level": level}
        })

    # 2) ward / district / province candidates (as before)
    # wards
    ward_pool = normalized_ward_map.get(span_norm, wards_list)
    for name, score in match_fuzzy_topk(span, ward_pool, top_k=top_k):
        candidates.append({"type": "ward", "name": name, "score": score, "details": ward_to_districts.get(name, [])})
    # districts
    district_pool = normalized_district_map.get(span_norm, districts_list)
    for name, score in match_fuzzy_topk(span, district_pool, top_k=top_k):
        candidates.append({"type": "district", "name": name, "score": score, "details": district_to_provinces.get(name, [])})
    # provinces
    province_pool = normalized_province_map.get(span_norm, provinces_list)
    for name, score in match_fuzzy_topk(span, province_pool, top_k=top_k):
        candidates.append({"type": "province", "name": name, "score": score, "details": []})

    # sort by score desc
    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return candidates

# -------------------------------
# Integration: attempt to match combined neighbor spans to full-address first
# -------------------------------
def try_combine_and_match_full(span_list, i, full_addresses, threshold=70):
    """
    Try combine span i with next 1-2 spans to see if combined string matches a full-address (district, province).
    Return (matched_full_string, score, end_index) or (None,0,i)
    """
    best = (None, 0, i)
    # try i, i+1 ; i, i+1, i+2
    for l in (1,2,3):
        if i + l > len(span_list):
            break
        combined = ", ".join(span_list[i:i+l])  # join with comma to mimic "Quận X, Thành phố Y"
        top = process.extractOne(combined, full_addresses, scorer=fuzz.WRatio)
        if top:
            full_name, score, _ = top
            if score > best[1]:
                best = (full_name, score, i + l - 1)
    if best[0] and best[1] >= threshold:
        return best
    return (None, 0, i)

# -------------------------------
# Example integration in a resolver (simplified)
# -------------------------------
def resolve_with_full_address(span_list, wards_list, districts_list, provinces_list,
                              ward_to_districts, district_to_provinces,
                              normalized_ward_map, normalized_district_map, normalized_province_map,
                              full_addresses, full_map, full_threshold=75, verbose=True):
    resolved = []
    i = 0
    while i < len(span_list):
        span = span_list[i]
        # 1) try combine spans to match a full-address exactly/fuzzy first
        full_name, score, end_idx = try_combine_and_match_full(span_list, i, full_addresses, threshold=full_threshold)
        if full_name:
            # accept full match: extract district/province from full_map
            district, province, level = full_map[full_name]
            resolved.append({'ward': None, 'districts': [district], 'provinces': [province], 'matched_full': full_name, 'score': score})
            if verbose:
                print(f"Combined spans [{i}-{end_idx}] -> matched FULL: {full_name} (score={score})")
            i = end_idx + 1
            continue

        # 2) otherwise generate candidates (including single-span full hits) and greedy choose best (as before)
        cands = gen_candidates_with_full(span, wards_list, districts_list, provinces_list,
                                         ward_to_districts, district_to_provinces,
                                         normalized_ward_map, normalized_district_map, normalized_province_map,
                                         full_addresses, full_map, top_k=6)
        # pick top candidate with heuristics (example)
        best = None
        best_sc = -999
        for c in cands[:10]:
            sc = c['score']
            # prefer full_address/district if score strong
            if c['type']=='full_address':
                sc += 15
            # penalize weak fuzzy raw
            if c['score'] < 50:
                sc -= 20
            if sc > best_sc:
                best_sc = sc
                best = c

        if best and best_sc >= 60:
            if best['type']=='ward':
                districts = list({d for d,_ in best['details']})
                provinces = list({p for _,p in best['details']})
                resolved.append({'ward': best['name'], 'districts': districts, 'provinces': provinces, 'score': best_sc})
            elif best['type']=='district' or best['type']=='full_address':
                if best['type']=='full_address':
                    d = best['details']['district']
                    p = best['details']['province']
                    resolved.append({'ward': None, 'districts': [d], 'provinces': [p], 'matched_full': best['name'], 'score': best_sc})
                else:
                    resolved.append({'ward': None, 'districts': [best['name']], 'provinces': best['details'], 'score': best_sc})
            else:
                resolved.append({'ward': None, 'districts': [], 'provinces': [best['name']], 'score': best_sc})
        else:
            resolved.append({'ward': None, 'districts': [], 'provinces': [], 'score': best_sc if best else None})
        i += 1

    return resolved

# -------------------------------
# Notes / How this helps
# -------------------------------
# 1) Matching the combined spans (e.g. "Quận Ba Đình, Thành phố Hà Nội" or "Ba Đình Hà Nội") against full_addresses
#    gives strong disambiguation: if a full-address matches well, you know exact district+province.
# 2) Even if NER split "Quận Ba Đình" and "Hà Nội" into two spans, try_combine_and_match_full will combine neighbor spans
#    and detect the full-address with higher confidence.
# 3) full_addresses should be normalized (you can store also lowercased normalized variants) and you can tune full_threshold.
# 4) Use full-address matching first, then fallback to ward/district/province component matching.
#
# Tuning tips:
# - full_threshold: 70-85 (higher => require stronger match to accept combined full address)
# - boost full-address candidate scores so they are preferred over single-component fuzzy hits
# - if your full list includes levels, you can prefer matching only those levels (e.g. when span has "Quận" prefix)


In [ ]:

# full_addresses, full_map, district_to_provinces, wards_list, districts_list, provinces_list
# normalized_ward_map, normalized_district_map, normalized_province_map
# ward_to_districts

In [124]:
# Giả sử bạn đã load:
# full_addresses, full_map, district_to_provinces, wards_list, districts_list, provinces_list
# normalized_ward_map, normalized_district_map, normalized_province_map
# ward_to_districts

# Ví dụ câu cần test
text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội"

# Giả sử NER đã tách các span LOCATION
ner_results = [
    {'entity_group': 'LOCATION', 'score': 0.99, 'word': 'xã Lĩnh Toại', 'start': 23, 'end': 35},
    {'entity_group': 'LOCATION', 'score': 0.99, 'word': 'huyện Hà Trung', 'start': 37, 'end': 51},
    {'entity_group': 'LOCATION', 'score': 0.99, 'word': 'tỉnh Thanh Hóa', 'start': 53, 'end': 67},
    {'entity_group': 'LOCATION', 'score': 0.99, 'word': 'xã Liễu Giai', 'start': 71, 'end': 83},
    {'entity_group': 'LOCATION', 'score': 0.93, 'word': 'Ba Đình', 'start': 84, 'end': 91},
    {'entity_group': 'LOCATION', 'score': 0.99, 'word': 'Hà Nội', 'start': 92, 'end': 98},
]

# Lấy danh sách span LOCATION để feed vào resolver
span_list = [r['word'] for r in ner_results]

# Chạy resolver với full-address matching
resolved = resolve_with_full_address(
    span_list,
    wards_list, districts_list, provinces_list,
    ward_to_districts, district_to_provinces,
    normalized_ward_map, normalized_district_map, normalized_province_map,
    full_addresses, full_map,
    full_threshold=75,  # threshold có thể điều chỉnh
    verbose=True
)

# In kết quả cuối cùng
for r in resolved:
    print(r)


NameError: name 'full_addresses' is not defined

In [129]:
from rapidfuzz import fuzz, process
from collections import defaultdict
import logging
import re

# -------------------------------
# Logging
# -------------------------------
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# -------------------------------
# Hàm tiện ích
# -------------------------------
def remove_prefix(name):
    """Loại bỏ tiền tố/suffix hành chính phổ biến"""
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

def _load_places_file(file_path):
    if not file_path:
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# -------------------------------
# Fuzzy matching
# -------------------------------
def match_fuzzy_topk(span, candidate_list, top_k=5, scorer=fuzz.WRatio):
    if not candidate_list:
        return []
    results = process.extract(span, candidate_list, scorer=scorer, limit=top_k)
    return [(r[0], r[1]) for r in results]

def select_best_candidate(span, top_candidates):
    span_tokens = set(span.lower().strip().split())
    best_score = -1
    best_candidate = None
    for cand, score in top_candidates:
        cand_tokens = set(cand.lower().strip().split())
        token_overlap = len(span_tokens & cand_tokens)
        combined_score = score + 20 * token_overlap
        logger.info(f"Candidate: '{cand}', fuzzy score: {score}, token overlap: {token_overlap}, combined: {combined_score}")
        if combined_score > best_score:
            best_score = combined_score
            best_candidate = cand
    return best_candidate

# -------------------------------
# Build dict từ file
# -------------------------------
def build_mappings(file_path):
    ward_to_districts = defaultdict(list)  # ward gốc -> list of (district, province)
    district_to_provinces = defaultdict(set)  # district -> set(province)
    normalized_ward_map = defaultdict(list)
    normalized_district_map = defaultdict(list)
    normalized_province_map = defaultdict(list)

    with open(file_path, "r", encoding="utf-8") as f:
        next(f)  # bỏ header
        for line in f:
            line = line.strip()
            if not line:
                continue
            ward, district, province = line.split("\t")

            # ward -> list (district, province)
            ward_to_districts[ward].append((district, province))
            # district -> province
            district_to_provinces[district].add(province)

            # build normalized maps
            normalized_ward_map[remove_prefix(ward)].append(ward)
            normalized_district_map[remove_prefix(district)].append(district)
            normalized_province_map[remove_prefix(province)].append(province)

    # chuyển set sang list
    district_to_provinces = {d: list(provs) for d, provs in district_to_provinces.items()}

    return ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map

# -------------------------------
# correct_address với normalized map
# -------------------------------
def correct_address(span, wards_list, districts_list, provinces_list,
                    ward_to_districts, district_to_provinces,
                    normalized_ward_map, normalized_district_map, normalized_province_map,
                    top_k=5, verbose=True):

    span_norm = remove_prefix(span)
    if verbose:
        logger.info(f"\nProcessing span: '{span}' -> normalized: '{span_norm}'")

    # 1️⃣ Match ward
    candidate_wards = normalized_ward_map.get(span_norm, wards_list)
    topk_wards = match_fuzzy_topk(span, candidate_wards, top_k=top_k)
    if verbose:
        # In kèm district/province tương ứng
        topk_info = [(w, s, ward_to_districts.get(w, [])) for w, s in topk_wards]
        logger.info(f"Top-K wards with districts/provinces: {topk_info}")

    ward = select_best_candidate(span, topk_wards) if topk_wards else None
    if ward:
        dp_list = ward_to_districts.get(ward, [])
        districts = [d for d, _ in dp_list]
        provinces = [p for _, p in dp_list]
        districts = list(set(districts))
        provinces = list(set(provinces))
        if verbose:
            logger.info(f"Selected ward: {ward}, districts: {districts}, provinces: {provinces}")
        return {'ward': ward, 'districts': districts, 'provinces': provinces}

    # 2️⃣ Match district
    candidate_districts = normalized_district_map.get(span_norm, districts_list)
    topk_districts = match_fuzzy_topk(span, candidate_districts, top_k=top_k)
    if verbose:
        topk_info = [(d, s, district_to_provinces.get(d, [])) for d, s in topk_districts]
        logger.info(f"Top-K districts with provinces: {topk_info}")

    district = select_best_candidate(span, topk_districts) if topk_districts else None
    if district:
        provinces = district_to_provinces.get(district, [])
        if verbose:
            logger.info(f"Selected district: {district}, provinces: {provinces}")
        return {'ward': None, 'districts': [district], 'provinces': provinces}

    # 3️⃣ Match province
    candidate_provinces = normalized_province_map.get(span_norm, provinces_list)
    topk_provinces = match_fuzzy_topk(span, candidate_provinces, top_k=top_k)
    province = select_best_candidate(span, topk_provinces) if topk_provinces else None
    if verbose:
        logger.info(f"Top-K provinces: {topk_provinces}")
        logger.info(f"Selected province: {province}")
    return {'ward': None, 'districts': [], 'provinces': [province] if province else []}

# -------------------------------
# Ví dụ test
# -------------------------------
if __name__ == "__main__":
    file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"

    # Build dict
    ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map = build_mappings(file_path)

    wards_list = list(ward_to_districts.keys())
    districts_list = list(district_to_provinces.keys())
    provinces_list = list({p for plist in district_to_provinces.values() for p in plist})

    test_spans = ["\Bưu gửi được chuyển đến: Ms. Huệ –, Công ty TNHH TM Ngọc Quê. Địa chỉ: Km82 Quốc lộ 1a, thôn Rừng Dông, xã Hữu Lũng, tỉnh Ninh Thuậtttt. Điện thoại liên hệ: 0985 378168."]

    test_spans = [
        "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội"
    ]

    for span in test_spans:
        corrected = correct_address(span, wards_list, districts_list, provinces_list,
                                    ward_to_districts, district_to_provinces,
                                    normalized_ward_map, normalized_district_map, normalized_province_map,
                                    verbose=True)
        logger.info(f"Final corrected: {corrected}")



Processing span: 'Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội' -> normalized: 'Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội'
Top-K wards with districts/provinces: [('Phường Liễu Giai', 85.5, [('Quận Ba Đình', 'Thành phố Hà Nội')]), ('Phường Ngọc Hà', 85.5, [('Quận Ba Đình', 'Thành phố Hà Nội'), ('Thành phố Hà Giang', 'Tỉnh Hà Giang')]), ('Phường Hàng Mã', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')]), ('Phường Hàng Buồm', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')]), ('Phường Hàng Đào', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')])]
Candidate: 'Phường Liễu Giai', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Ngọc Hà', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Hàng Mã', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Hàng Buồm', fuzzy score: 85.5, token overlap: 1, combined: 105.

In [74]:
def _load_places_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

In [ ]:
from rapidfuzz import fuzz, process
import logging

# -------------------------------
# Thiết lập logging
# -------------------------------
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# -------------------------------
# Các hàm cơ bản
# -------------------------------

def match_fuzzy_topk(span, candidate_list, top_k=5, scorer=fuzz.WRatio):
    """Trả về top-K candidate fuzzy match với score"""
    if not candidate_list:
        return []
    results = process.extract(span, candidate_list, scorer=scorer, limit=top_k)
    return [(r[0], r[1]) for r in results]

def select_best_candidate(span, top_candidates):
    """
    Chọn candidate tốt nhất trong Top-K bằng cách kết hợp:
    - Levenshtein ratio
    - Token overlap
    """
    span_tokens = set(span.lower().strip().split())
    best_score = -1
    best_candidate = None

    for cand, score in top_candidates:
        cand_tokens = set(cand.lower().strip().split())
        token_overlap = len(span_tokens & cand_tokens)
        combined_score = score + 20 * token_overlap  # trọng số token overlap
        logger.info(f"Candidate: '{cand}', fuzzy score: {score}, token overlap: {token_overlap}, combined: {combined_score}")
        if combined_score > best_score:
            best_score = combined_score
            best_candidate = cand

    return best_candidate

# -------------------------------
# Cascade inference chính
# -------------------------------

def correct_address(span, wards_list, districts_list, provinces_list,
                    ward_to_district, district_to_province,
                    top_k=5, threshold=80):
    logger.info(f"\nProcessing span: '{span}'")

    # 1️⃣ Thử match ward
    topk_wards = match_fuzzy_topk(span, wards_list, top_k=top_k)
    logger.info(f"Top-K wards: {topk_wards}")
    ward = select_best_candidate(span, topk_wards) if topk_wards else None

    if ward:
        district = ward_to_district.get(ward)
        province = district_to_province.get(district) if district else None
        logger.info(f"Selected ward: {ward}, inferred district: {district}, province: {province}")
        return {'ward': ward, 'district': district, 'province': province}

    # 2️⃣ Thử match district
    topk_districts = match_fuzzy_topk(span, districts_list, top_k=top_k)
    logger.info(f"Top-K districts: {topk_districts}")
    district = select_best_candidate(span, topk_districts) if topk_districts else None

    if district:
        province = district_to_province.get(district)
        logger.info(f"Selected district: {district}, inferred province: {province}")
        return {'ward': None, 'district': district, 'province': province}

    # 3️⃣ Thử match province
    topk_provinces = match_fuzzy_topk(span, provinces_list, top_k=top_k)
    logger.info(f"Top-K provinces: {topk_provinces}")
    province = select_best_candidate(span, topk_provinces) if topk_provinces else None

    logger.info(f"Selected province: {province}")
    return {'ward': None, 'district': None, 'province': province}

# -------------------------------
# Ví dụ test
# -------------------------------
if __name__ == "__main__":
    # wards_list = ["Phường 1_Ba Đình_Hà Nội", "Phường 1_Quận 1_Hồ Chí Minh", "Phúc Xá_Ba Đình_Hà Nội"]
    # districts_list = ["Ba Đình", "Quận 1", "Thanh Trì"]
    # provinces_list = ["Hà Nội", "Hồ Chí Minh", "Ninh Thuận"]


    wards_file = "/home/nampv1/projects/asr/asr_ft/postprocessing/address/wards.txt"
    # wards_file = None
    districts_file = "/home/nampv1/projects/asr/asr_ft/postprocessing/address/districts.txt"
    districts_file = None
    provinces_file = "/home/nampv1/projects/asr/asr_ft/postprocessing/address/provinces.txt"



    wards_list = _load_places_file(wards_file)
    districts_list = _load_places_file(districts_file)
    provinces_list = _load_places_file(provinces_file)


    ward_to_district = {
        "Phường 1_Ba Đình_Hà Nội": "Ba Đình",
        "Phường 1_Quận 1_Hồ Chí Minh": "Quận 1",
        "Phúc Xá_Ba Đình_Hà Nội": "Ba Đình"
    }

    district_to_province = {
        "Ba Đình": "Hà Nội",
        "Quận 1": "Hồ Chí Minh",
        "Thanh Trì": "Hà Nội"
    }

    test_spans = ["Phuc Xa", "Phuong 1", "Thanh Tri", "Ninh Thuật"]

    for span in test_spans:
        corrected = correct_address(span, wards_list, districts_list, provinces_list,
                                    ward_to_district, district_to_province)
        logger.info(f"Final corrected: {corrected}")



Processing span: 'Phuc Xa'
Top-K wards: [('Phúc Xá_Ba Đình_Hà Nội', 69.23076923076923), ('Phường 1_Ba Đình_Hà Nội', 40.0), ('Phường 1_Quận 1_Hồ Chí Minh', 40.0)]
Candidate: 'Phúc Xá_Ba Đình_Hà Nội', fuzzy score: 69.23076923076923, token overlap: 0, combined: 69.23076923076923
Candidate: 'Phường 1_Ba Đình_Hà Nội', fuzzy score: 40.0, token overlap: 0, combined: 40.0
Candidate: 'Phường 1_Quận 1_Hồ Chí Minh', fuzzy score: 40.0, token overlap: 0, combined: 40.0
Selected ward: Phúc Xá_Ba Đình_Hà Nội, inferred district: Ba Đình, province: Hà Nội
Final corrected: {'ward': 'Phúc Xá_Ba Đình_Hà Nội', 'district': 'Ba Đình', 'province': 'Hà Nội'}

Processing span: 'Phuong 1'
Top-K wards: [('Phường 1_Ba Đình_Hà Nội', 67.5), ('Phường 1_Quận 1_Hồ Chí Minh', 67.5), ('Phúc Xá_Ba Đình_Hà Nội', 41.53846153846154)]
Candidate: 'Phường 1_Ba Đình_Hà Nội', fuzzy score: 67.5, token overlap: 0, combined: 67.5
Candidate: 'Phường 1_Quận 1_Hồ Chí Minh', fuzzy score: 67.5, token overlap: 0, combined: 67.5
Candidate

In [122]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from rapidfuzz import fuzz, process
from collections import defaultdict
import logging
import re
import itertools

# -------------------------------
# Logging
# -------------------------------
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# -------------------------------
# Utils (same as before)
# -------------------------------
def remove_prefix(name):
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name, flags=re.I)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name, flags=re.I)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name, flags=re.I)
    return name.strip()

def get_prefix_type(name):
    if re.match(r"^(Xã|Phường|Thị trấn)\s+", name, flags=re.I):
        return "ward"
    if re.match(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", name, flags=re.I):
        return "district"
    if re.match(r"^(Tỉnh|Thành phố)\s+", name, flags=re.I):
        return "province"
    return None

def match_fuzzy_topk(span, candidate_list, top_k=8, scorer=fuzz.WRatio):
    if not candidate_list:
        return []
    results = process.extract(span, candidate_list, scorer=scorer, limit=top_k)
    return [(r[0], r[1]) for r in results]

# -------------------------------
# Build mappings (from your file)
# -------------------------------
def build_mappings(file_path):
    ward_to_districts = defaultdict(list)
    district_to_provinces = defaultdict(set)
    normalized_ward_map = defaultdict(list)
    normalized_district_map = defaultdict(list)
    normalized_province_map = defaultdict(list)

    with open(file_path, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line:
                continue
            ward, district, province = line.split("\t")
            ward_to_districts[ward].append((district, province))
            district_to_provinces[district].add(province)
            normalized_ward_map[remove_prefix(ward)].append(ward)
            normalized_district_map[remove_prefix(district)].append(district)
            normalized_province_map[remove_prefix(province)].append(province)

    # sets -> lists
    district_to_provinces = {d: list(ps) for d, ps in district_to_provinces.items()}
    return ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map

# -------------------------------
# Candidate generation
# -------------------------------
def gen_candidates_for_span(span, wards_list, districts_list, provinces_list,
                            ward_to_districts, district_to_provinces,
                            normalized_ward_map, normalized_district_map, normalized_province_map,
                            top_k=8):
    """
    Trả về list candidate: each = {
      'type': 'ward'/'district'/'province',
      'name': name_in_db,
      'score': fuzzy_score (0-100),
      'details': for ward -> list of (district,province); for district -> provinces list
    }
    """
    span_norm = remove_prefix(span)
    pref = get_prefix_type(span)

    candidates = []

    # ward candidates (priority if pref=='ward')
    ward_pool = normalized_ward_map.get(span_norm, wards_list)
    topk_wards = match_fuzzy_topk(span, ward_pool, top_k=top_k)
    for name, score in topk_wards:
        candidates.append({
            'type': 'ward', 'name': name, 'score': score,
            'details': ward_to_districts.get(name, [])
        })

    # district candidates
    district_pool = normalized_district_map.get(span_norm, districts_list)
    topk_districts = match_fuzzy_topk(span, district_pool, top_k=top_k)
    for name, score in topk_districts:
        candidates.append({
            'type': 'district', 'name': name, 'score': score,
            'details': district_to_provinces.get(name, [])
        })

    # province candidates
    province_pool = normalized_province_map.get(span_norm, provinces_list)
    topk_provinces = match_fuzzy_topk(span, province_pool, top_k=top_k)
    for name, score in topk_provinces:
        candidates.append({
            'type': 'province', 'name': name, 'score': score,
            'details': []
        })

    # if prefix exists, boost candidates of that type
    if pref:
        for c in candidates:
            if c['type'] == pref:
                c['score'] += 15  # heuristic boost for explicit prefixes

    # normalize scores to 0-100 and sort
    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return candidates

# -------------------------------
# Joint inference for hierarchical chunk (ward, district, province)
# -------------------------------
def resolve_hierarchical_chunk(span_sequence, candidates_per_span, ward_to_districts, district_to_provinces):
    """
    Try to jointly pick candidates for spans if they form a hierarchical sequence
    (e.g., [ward_span, district_span, province_span]).
    We'll enumerate reasonable top-K combos and choose highest-scoring consistent combo.

    Return list of resolved dicts (same length as span_sequence), or None if not confident.
    """
    n = len(span_sequence)
    # limit top candidates to some K to enumerate
    TOP_CAND_LIMIT = 5
    top_lists = [cands[:TOP_CAND_LIMIT] for cands in candidates_per_span]

    best_combo = None
    best_score = -1
    # enumerate cartesian product (bounded by TOP_CAND_LIMIT^n)
    for combo in itertools.product(*top_lists):
        score = 0.0
        consistent = True

        # accumulate base scores
        for c in combo:
            score += c['score']

        # enforce hierarchical consistency:
        # if there's a ward candidate and a district candidate in the combo, ward's districts must include district
        ward_candidates = [c for c in combo if c['type']=='ward']
        district_candidates = [c for c in combo if c['type']=='district']
        province_candidates = [c for c in combo if c['type']=='province']

        # for each ward candidate, ensure it is compatible with at least one district/province candidate in the combo (if present)
        for wc in ward_candidates:
            ward_dlist = [d for d,_ in wc['details']]
            # if a district candidate exists but not in ward_dlist -> penalize heavily / mark inconsistent
            if district_candidates:
                if not any(dc['name'] in ward_dlist for dc in district_candidates):
                    consistent = False
                    break
            # if a province candidate exists but ward's provinces don't include it -> inconsistent
            if province_candidates:
                ward_provs = [p for _,p in wc['details']]
                if not any(pc['name'] in ward_provs for pc in province_candidates):
                    consistent = False
                    break

        # also check district->province compatibility
        for dc in district_candidates:
            provs = district_to_provinces.get(dc['name'], [])
            if province_candidates and not any(pc['name'] in provs for pc in province_candidates):
                consistent = False
                break

        if not consistent:
            continue

        # add small bonus for cross-span agreement (if same district/province appears multiple times)
        # count agreements
        district_names = [c['name'] for c in combo if c['type']=='district']
        province_names = [c['name'] for c in combo if c['type']=='province']
        score += 5 * (len(district_names) - len(set(district_names)))  # duplicates -> bonus
        score += 3 * (len(province_names) - len(set(province_names)))

        if score > best_score:
            best_score = score
            best_combo = combo

    if best_combo and best_score > 0:
        # convert to results
        results = []
        for c in best_combo:
            if c['type']=='ward':
                districts = list({d for d,_ in c['details']})
                provinces = list({p for _,p in c['details']})
                results.append({'ward': c['name'], 'districts': districts, 'provinces': provinces})
            elif c['type']=='district':
                results.append({'ward': None, 'districts': [c['name']], 'provinces': district_to_provinces.get(c['name'], [])})
            else:
                results.append({'ward': None, 'districts': [], 'provinces': [c['name']]})
        return results
    return None

# -------------------------------
# Main context-aware resolver for span list
# -------------------------------
def resolve_spans_with_context(span_list, wards_list, districts_list, provinces_list,
                               ward_to_districts, district_to_provinces,
                               normalized_ward_map, normalized_district_map, normalized_province_map,
                               verbose=True):
    """
    span_list: list of LOC spans in order
    Returns list of resolution dicts aligned with spans.
    """
    # 1) generate candidates for each span
    all_candidates = []
    for span in span_list:
        cands = gen_candidates_for_span(span, wards_list, districts_list, provinces_list,
                                        ward_to_districts, district_to_provinces,
                                        normalized_ward_map, normalized_district_map, normalized_province_map)
        all_candidates.append(cands)
        if verbose:
            logger.info(f"\nCandidates for '{span}':")
            for c in cands[:8]:
                logger.info(f"  {c['type']}: {c['name']} (score={c['score']}) -> details={c['details']}")

    # 2) detect explicit hierarchical groups in span_list
    # heuristic: runs of spans that include a prefix type sequence (ward -> district -> province)
    groups = []  # list of (start, end) inclusive
    i = 0
    while i < len(span_list):
        # try to form group of up to length 3: ward,district,province in order (prefix presence)
        j = i
        group_spans = [span_list[j]]
        prefixes = [get_prefix_type(span_list[j])]
        while j+1 < len(span_list) and len(group_spans) < 3:
            j += 1
            group_spans.append(span_list[j])
            prefixes.append(get_prefix_type(span_list[j]))
        # if group has at least two non-None prefixes and they are in descending granularity, accept it
        non_none = [p for p in prefixes if p is not None]
        if len(non_none) >= 2:
            groups.append((i, j))
            i = j + 1
        else:
            i += 1

    # fallback: if no groups detected, also try sequential triples separated by commas in original text could be used.
    # For simplicity here use groups as detected; others handled by greedy below.

    resolved = [None] * len(span_list)

    # 3) resolve groups with joint inference
    for (s, e) in groups:
        span_seq = span_list[s:e+1]
        candidates_seq = all_candidates[s:e+1]
        joint = resolve_hierarchical_chunk(span_seq, candidates_seq, ward_to_districts, district_to_provinces)
        if joint:
            for idx, res in enumerate(joint):
                resolved[s+idx] = res

    # 4) greedy resolve remaining spans left-to-right with context boosting
    last_districts = []
    last_provinces = []
    for idx, span in enumerate(span_list):
        if resolved[idx] is not None:
            # update context
            last_districts = resolved[idx].get('districts', []) or last_districts
            last_provinces = resolved[idx].get('provinces', []) or last_provinces
            continue

        cands = all_candidates[idx]
        best = None
        best_score = -1
        for c in cands:
            score = c['score']
            # boost if compatible with context
            if c['type']=='ward':
                # if ward's districts intersect last_districts, boost heavily
                w_d = [d for d,_ in c['details']]
                if last_districts and any(d in last_districts for d in w_d):
                    score += 25
                # if ward's provinces intersect last_provinces
                w_p = [p for _,p in c['details']]
                if last_provinces and any(p in last_provinces for p in w_p):
                    score += 20
            elif c['type']=='district':
                if last_provinces and any(p in last_provinces for p in district_to_provinces.get(c['name'], [])):
                    score += 25
            elif c['type']=='province':
                if last_provinces and c['name'] in last_provinces:
                    score += 25

            # penalize low raw fuzzy scores
            if c['score'] < 50:
                score -= 20

            if score > best_score:
                best_score = score
                best = c

        # threshold to accept
        if best and best_score >= 60:
            if best['type']=='ward':
                districts = list({d for d,_ in best['details']})
                provinces = list({p for _,p in best['details']})
                resolved[idx] = {'ward': best['name'], 'districts': districts, 'provinces': provinces}
                last_districts = districts
                last_provinces = provinces
            elif best['type']=='district':
                resolved[idx] = {'ward': None, 'districts': [best['name']], 'provinces': district_to_provinces.get(best['name'], [])}
                last_districts = [best['name']]
                last_provinces = district_to_provinces.get(best['name'], [])
            else:
                resolved[idx] = {'ward': None, 'districts': [], 'provinces': [best['name']]}
                last_provinces = [best['name']]
        else:
            # fallback: leave empty
            resolved[idx] = {'ward': None, 'districts': [], 'provinces': []}

    return resolved

# -------------------------------
# Example run
# -------------------------------
if __name__ == "__main__":
    # Load NER (you probably already have this loaded)
    tokenizer = AutoTokenizer.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
    model = AutoModelForTokenClassification.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
    nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

    # Build mappings from your file
    file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"
    ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map = build_mappings(file_path)
    wards_list = list(ward_to_districts.keys())
    districts_list = list(district_to_provinces.keys())
    provinces_list = list({p for plist in district_to_provinces.values() for p in plist})

    text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội"
    ner_results = nlp(text)
    span_list = [ent['word'] for ent in ner_results if ent['entity_group'] in ('LOC','LOCATION')]

    logger.info("NER spans: %s", span_list)
    resolved = resolve_spans_with_context(span_list, wards_list, districts_list, provinces_list,
                                         ward_to_districts, district_to_provinces,
                                         normalized_ward_map, normalized_district_map, normalized_province_map,
                                         verbose=True)

    for s, r in zip(span_list, resolved):
        logger.info("Span: '%s' -> %s", s, r)


Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
NER spans: ['xã Lĩnh Toại', 'huyện Hà Trung', 'tỉnh Thanh Hóa', 'xã Liễu Giai', 'Ba Đình', 'Hà Nội']

Candidates for 'xã Lĩnh Toại':
  ward: Xã Lĩnh Toại (score=106.66666666666666) -> details=[('Huyện Hà Trung', 'Tỉnh Thanh Hóa')]
  district: Thị xã Thuận Thành (score=85.5) -> details=['Tỉnh Bắc Ninh']
  district: Thị xã Hồng Lĩnh (score=70.0) -> details=['Tỉnh Hà Tĩnh']
  district: Thị xã Kỳ Anh (score=53.2) -> details=['Tỉnh Hà Tĩnh']
  district: Thị xã An Khê (score=53.2) -> details=['Tỉnh Gia Lai']
  district: Thị xã Phước Long (score=52.41379310344827) -> details=['Tỉnh Bình Phước']
  district: Thị xã Chũ (score=51.81818181818182) -> details=['Tỉnh Bắc Giang']
  district: Thị xã Sơn Tây (score=51.15384615384615) -> details=['Thành phố Hà Nội']

Candidates for 'huyện Hà Trung':
  district: Huyện Hà Trung (score=107.857

In [102]:
len(wards_list), len(districts_list), len(provinces_list)

(10047, 696, 63)

In [59]:
wards_file = "/home/nampv1/projects/asr/asr_ft/postprocessing/address/wards.txt"
wards = _load_places_file(wards_file)
wards[:10]

['Phúc Xá',
 'Trúc Bạch',
 'Vĩnh Phúc',
 'Cống Vị',
 'Liễu Giai',
 'Quán Thánh',
 'Ngọc Hà',
 'Điện Biên',
 'Đội Cấn',
 'Ngọc Khánh']

In [53]:
def _load_places_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


In [60]:
from rapidfuzz import fuzz, process

def match_ward(span, wards_list, threshold=80):
    """
    span: string input từ NER/ASR
    wards_list: list of canonical ward names
    returns: best ward or None
    """
    # normalize
    def norm(s): 
        import unicodedata
        s = s.strip().lower()
        s = unicodedata.normalize("NFKD", s)
        return "".join(c for c in s if not unicodedata.combining(c))

    norm_span = norm(span)
    
    best = None
    best_score = 0

    for ward in wards_list:
        ward_norm = norm(ward)
        lev = fuzz.ratio(norm_span, ward_norm)
        wr = fuzz.WRatio(norm_span, ward_norm)
        score = 1.5*lev + 0.5*wr  # ưu tiên Levenshtein

        if score > best_score:
            best_score = score
            best = ward

    if best_score >= threshold:
        return best
    return None


def match_ward_no_norm(span, wards_list, threshold=80):
    best = None
    best_score = 0
    for ward in wards_list:
        lev = fuzz.ratio(span, ward)
        wr = fuzz.WRatio(span, ward)
        score = 1.5*lev + 0.5*wr
        if score > best_score:
            best_score = score
            best = ward
    if best_score >= threshold:
        return best
    return None


In [62]:
# # Ví dụ danh sách wards
# wards = [
#     "Phúc Xá",
#     "Trúc Bạch",
#     "Ninh Thuận",
#     "Phúc Lợi",
#     "Ba Đình"
# ]

# Test các span ASR
test_spans = [
    "Phuc Xa",      # đúng ward
    "Truc Bach",    # đúng ward
    "Ninh Thuật",   # lỗi chính tả, mong correct về "Ninh Thuận"
    "Phuc",         # quá ngắn, không confident
    "Ba Dình"       # trùng tên quận/ward, test
]

for span in test_spans:
    match = match_ward_no_norm(span, wards, threshold=80)
    print(f"Input: '{span}' -> Matched ward: {match}")

Input: 'Phuc Xa' -> Matched ward: Phúc Xá
Input: 'Truc Bach' -> Matched ward: Trúc Bạch
Input: 'Ninh Thuật' -> Matched ward: Minh Thuận
Input: 'Phuc' -> Matched ward: Phố Lu
Input: 'Ba Dình' -> Matched ward: Ba Đình


In [67]:
from rapidfuzz import process, fuzz

def get_topk_wards(span, wards_list, top_k=5, scorer=fuzz.WRatio):
    """
    Lấy top-k wards gần giống span theo fuzzy matching.

    Args:
        span (str): string input từ ASR/NER
        wards_list (list): danh sách wards canonical
        top_k (int): số lượng candidate trả về
        scorer: hàm scoring từ RapidFuzz (mặc định WRatio)

    Returns:
        list of tuples: [(ward_name, score), ...] sắp xếp theo score giảm dần
    """
    if not wards_list:
        return []

    # RapidFuzz process.extract trả về list [(candidate, score, idx)]
    results = process.extract(span, wards_list, scorer=scorer, limit=top_k)
    # Bỏ idx, chỉ giữ name + score
    top_candidates = [(r[0], r[1]) for r in results]
    return top_candidates


In [70]:
# wards = [
#     "Phường 1_Ba Đình_Hà Nội",
#     "Phường 1_Quận 1_Hồ Chí Minh",
#     "Phường 2_Ba Đình_Hà Nội",
#     "Phúc Xá_Ba Đình_Hà Nội"
# ]

test_spans = [
    "Ba Dinh",
    "Phuong 1",
    "Phuc Xa",
    "Phuong 2"
]

for span in test_spans:
    topk = get_topk_wards(span, wards, top_k=3)
    print(f"Input: '{span}' -> Top-K wards: {topk}")


Input: 'Ba Dinh' -> Top-K wards: [('Ba Dinh', 100.0), ('Ba', 90.0), ('Ba Vinh', 85.71428571428572)]
Input: 'Phuong 1' -> Top-K wards: [('Xuân Quang 1', 85.5), ('Thường Phước 1', 85.5), ('Phong Dụ Thượng', 77.14285714285715)]
Input: 'Phuc Xa' -> Top-K wards: [('Phúc Xá', 71.42857142857143), ('Phúc La', 71.42857142857143), ('Đa Phúc', 67.85714285714286)]
Input: 'Phuong 2' -> Top-K wards: [('Xuân Quang 2', 85.5), ('Thường Phước 2', 85.5), ('Phong Dụ Thượng', 77.14285714285715)]


In [82]:
import re
from collections import defaultdict

def remove_prefix(name):
    """
    Loại bỏ các tiền tố/suffix hành chính phổ biến: Xã, Phường, Thị trấn, Huyện, Quận, Thị xã, Thành phố
    """
    # Loại bỏ tiền tố
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    # Loại bỏ hậu tố
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

ward_to_district = defaultdict(list)
district_to_province = defaultdict(list)

file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"

with open(file_path, "r", encoding="utf-8") as f:
    next(f)  # bỏ header
    for line in f:
        line = line.strip()
        if not line:
            continue
        ward, district, province = line.split("\t")
        ward_clean = remove_prefix(ward)
        district_clean = remove_prefix(district)
        province_clean = remove_prefix(province)

        if district_clean not in ward_to_district[ward_clean]:
            ward_to_district[ward_clean].append(district_clean)
        if province_clean not in district_to_province[district_clean]:
            district_to_province[district_clean].append(province_clean)

# Ví dụ kiểm tra
ward = "Phường Phúc Xá"
ward_clean = remove_prefix(ward)
districts = ward_to_district.get(ward_clean, [])
for d in districts:
    provinces = district_to_province.get(d, [])
    for p in provinces:
        print(f"Xã/Phường: {ward_clean}, Quận/Huyện: {d}, Tỉnh/TP: {p}")


Xã/Phường: Phúc Xá, Quận/Huyện: Ba Đình, Tỉnh/TP: Hà Nội


In [97]:
from collections import defaultdict
import re

def remove_prefix(name):
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"

# Đếm tổng số ward trong file
wards_before_count = 0
with open(file_path, "r", encoding="utf-8") as f:
    next(f)
    for line in f:
        if line.strip():
            wards_before_count += 1

print(f"Tổng số ward trong file (trước khi tạo dict): {wards_before_count}")

# Tạo dict giữ tất cả bản ghi, không bỏ qua trùng lặp
ward_records = defaultdict(list)  # ward gốc -> list of (district, province)
normalized_ward_map = defaultdict(list)

with open(file_path, "r", encoding="utf-8") as f:
    next(f)
    for line in f:
        line = line.strip()
        if not line:
            continue
        ward, district, province = line.split("\t")
        ward_norm = remove_prefix(ward)

        ward_records[ward].append((district, province))
        normalized_ward_map[ward_norm].append(ward)

# Đếm tổng số ward theo bản ghi (không bỏ trùng)
wards_after_count = sum(len(records) for records in ward_records.values())
print(f"Tổng số ward theo bản ghi (sau khi tạo dict, giữ tên gốc): {wards_after_count}")


Tổng số ward trong file (trước khi tạo dict): 10047
Tổng số ward theo bản ghi (sau khi tạo dict, giữ tên gốc): 10047


In [111]:
ward_to_districts

{'Phường Phúc Xá': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Trúc Bạch': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Vĩnh Phúc': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Cống Vị': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Liễu Giai': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Quán Thánh': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Ngọc Hà': [('Quận Ba Đình', 'Thành phố Hà Nội'),
  ('Thành phố Hà Giang', 'Tỉnh Hà Giang')],
 'Phường Điện Biên': [('Quận Ba Đình', 'Thành phố Hà Nội'),
  ('Thành phố Thanh Hóa', 'Tỉnh Thanh Hóa')],
 'Phường Đội Cấn': [('Quận Ba Đình', 'Thành phố Hà Nội'),
  ('Thành phố Tuyên Quang', 'Tỉnh Tuyên Quang')],
 'Phường Ngọc Khánh': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Kim Mã': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Giảng Võ': [('Quận Ba Đình', 'Thành phố Hà Nội')],
 'Phường Thành Công': [('Quận Ba Đình', 'Thành phố Hà Nội'),
  ('Thành phố Buôn Ma Thuột', 'Tỉnh Đắk Lắk')],
 'Phường Phúc Tân': [('Quận Hoàn

In [106]:
from collections import defaultdict
import re

def remove_prefix(name):
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

# tạo normalized map: tên đã remove_prefix -> list ward gốc
normalized_ward_map = defaultdict(list)
for ward, dp_list in ward_to_districts.items():
    ward_norm = remove_prefix(ward)
    normalized_ward_map[ward_norm].append(ward)

# tương tự nếu muốn normalized district hoặc province
normalized_district_map = defaultdict(list)
for district, provinces in district_to_provinces.items():
    district_norm = remove_prefix(district)
    normalized_district_map[district_norm].append(district)

normalized_province_map = defaultdict(list)
for province_list in district_to_provinces.values():
    for province in province_list:
        province_norm = remove_prefix(province)
        normalized_province_map[province_norm].append(province)


In [108]:
len(normalized_ward_map)

6988

In [110]:
# ward_to_districts: ward gốc -> list of (district, province)
ward_to_districts = dict(ward_records)  # đã là dạng ward -> list of (district, province)

# district_to_provinces: district -> list of province (loại trùng)
district_to_provinces = defaultdict(set)
for ward, dp_list in ward_records.items():
    for district, province in dp_list:
        district_to_provinces[district].add(province)

# chuyển set sang list
district_to_provinces = {d: list(provs) for d, provs in district_to_provinces.items()}

# kiểm tra ví dụ
example_ward = "Phúc Xá"
print(f"Ward '{example_ward}' maps to districts/provinces: {ward_to_districts.get(example_ward)}")

example_district = "Ba Đình"
print(f"District '{example_district}' maps to provinces: {district_to_provinces.get(example_district)}")


Ward 'Phúc Xá' maps to districts/provinces: None
District 'Ba Đình' maps to provinces: None


In [94]:
import re
from collections import defaultdict

def remove_prefix(name):
    """Loại bỏ các tiền tố/suffix hành chính phổ biến"""
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"

# Map normalized ward -> list of original wards
normalized_map = defaultdict(list)

with open(file_path, "r", encoding="utf-8") as f:
    next(f)  # bỏ header
    for line in f:
        line = line.strip()
        if not line:
            continue
        ward, district, province = line.split("\t")
        ward_norm = remove_prefix(ward)
        normalized_map[ward_norm].append(ward)

# Lọc những normalized_name có nhiều hơn 1 ward gốc
conflicts = {norm: originals for norm, originals in normalized_map.items() if len(originals) > 1}

print(f"Số ward bị gộp do remove_prefix: {len(conflicts)}\n")
for norm, originals in conflicts.items():
    print(f"{norm}: {originals}")


Số ward bị gộp do remove_prefix: 1201

Vĩnh Phúc: ['Phường Vĩnh Phúc', 'Xã Vĩnh Phúc', 'Xã Vĩnh Phúc']
Ngọc Hà: ['Phường Ngọc Hà', 'Phường Ngọc Hà']
Điện Biên: ['Phường Điện Biên', 'Phường Điện Biên']
Đội Cấn: ['Phường Đội Cấn', 'Phường Đội Cấn', 'Xã Đội Cấn']
Thành Công: ['Phường Thành Công', 'Xã Thành Công', 'Xã Thành Công', 'Xã Thành Công', 'Phường Thành Công', 'Xã Thành Công']
Phúc Tân: ['Phường Phúc Tân', 'Xã Phúc Tân']
Đồng Xuân: ['Phường Đồng Xuân', 'Xã Đồng Xuân', 'Phường Đồng Xuân']
Chương Dương: ['Phường Chương Dương', 'Xã Chương Dương']
Cửa Nam: ['Phường Cửa Nam', 'Phường Cửa Nam', 'Phường Cửa Nam']
Trần Hưng Đạo: ['Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Xã Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo', 'Phường Trần Hưng Đạo']
Phú Thượng: ['Phường Phú Thượng', 'Xã Phú Thượng', 'Phường Phú Thượng']
Nhật Tân: ['Phường Nhật Tân', 'Xã Nhật Tân']
Quảng An: ['Phường Quảng An', 'Xã Quả

In [89]:
len(ward_to_district), len(district_to_province)

(6988, 662)

In [83]:
ward_to_district

defaultdict(list,
            {'Phúc Xá': ['Ba Đình'],
             'Trúc Bạch': ['Ba Đình'],
             'Vĩnh Phúc': ['Ba Đình', 'Bắc Quang', 'Vĩnh Lộc'],
             'Cống Vị': ['Ba Đình'],
             'Liễu Giai': ['Ba Đình'],
             'Quán Thánh': ['Ba Đình'],
             'Ngọc Hà': ['Ba Đình', 'Hà Giang'],
             'Điện Biên': ['Ba Đình', 'Thanh Hóa'],
             'Đội Cấn': ['Ba Đình', 'Tuyên Quang', 'Tràng Định'],
             'Ngọc Khánh': ['Ba Đình'],
             'Kim Mã': ['Ba Đình'],
             'Giảng Võ': ['Ba Đình'],
             'Thành Công': ['Ba Đình',
              'Nguyên Bình',
              'Phổ Yên',
              'Thạch Thành',
              'Buôn Ma Thuột',
              'Gò Công Tây'],
             'Phúc Tân': ['Hoàn Kiếm', 'Phổ Yên'],
             'Đồng Xuân': ['Hoàn Kiếm', 'Thanh Ba', 'Phúc Yên'],
             'Hàng Mã': ['Hoàn Kiếm'],
             'Hàng Buồm': ['Hoàn Kiếm'],
             'Hàng Đào': ['Hoàn Kiếm'],
             'Hàng Bồ': ['

In [84]:
len(ward_to_district), len(district_to_province)

(6988, 662)

In [43]:
import requests
import json
import time

url = 'https://ai.vnpost.vn/vllm-openai-oss-143/v1/chat/completions'
headers = {
    'Content-Type': 'application/json'
}
data = {
    "model": "openai/gpt-oss-20b",
    "messages": [
        {"role": "system", "content": "Bạn là một chuyên gia trong nhận diện địa chỉ hành chính của Việt Nam."},
        {"role": "user", "content": "Hãy xác định địa chỉ hành chính trong văn bản sau (nếu có)"},
        {"role": "user", "content": "bà lê thục anh 039 260 0806 16 phan chu trinh ba vì hà nội"}
        
    ]
}

start_time = time.time()
response = requests.post(url, headers=headers, data=json.dumps(data))
end_time = time.time()

processing_time = end_time - start_time
print(f"Thời gian xử lý: {processing_time:.2f} giây")

result = response.json()
# print(response.json())


Thời gian xử lý: 2.41 giây


In [44]:
result = response.json()

# Giả sử API trả về theo cấu trúc "choices" giống OpenAI
if "choices" in result and len(result["choices"]) > 0:
    content = result["choices"][0]["message"]["content"]
    # print("Kết quả từ LLM:")
    print(content)
else:
    print("Không nhận được phản hồi hợp lệ từ API.")


Địa chỉ hành chính được rút ra trong văn bản là:

**16 Phan Chu Trinh, Quận Ba Vì, Hà Nội, Việt Nam**  

(Phân tích: “16 phan chu trinh” → số nhà & tên đường, “ba vì” → quận/ huyện trong thành phố Hà Nội).


In [29]:
response.json()

{'id': 'chatcmpl-abdf8bfc5ad64c978b78a3d26569e245',
 'object': 'chat.completion',
 'created': 1759133049,
 'model': 'openai/gpt-oss-20b',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '**Địa chỉ hành chính được xác định từ văn bản:**\n\n- **Thành phố**: Hà Nội  \n- **Quận**: Đống Đa  \n- **Đường và số nhà**: 123 Đường Láng  \n\n=> **Địa chỉ đầy đủ**: **123 Đường Láng, Quận Đống Đa, Thành phố Hà Nội**.',
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning_content': 'We must identify administrative address. Input: "123 Đường Láng, Quận Đống Đa, Hà Nội". It\'s a Vietnamese administrative address: street address, ward? Actually it says "Quận Đống Đa" (District Đống Đa) in Hanoi. No commune/ward. But "Đường Láng, Quận Đống Đa, Hà Nội" has street and district and city.\n\nWe need to answer: Identify administrative divisions present: Street, District, City (Province-level). So maybe wri

In [20]:
from rapidfuzz import fuzz, process

query = "Ninh Thuậttt"
cands = ["Ninh Thuận", "Bắc Ninh", "Ninh Thuận Province", "Hiền Ninh"]  # ví dụ

for c in cands:
    print("CAND:", c)
    print("  WRatio:", fuzz.WRatio(query, c))
    print("  Ratio (levenshtein):", fuzz.ratio(query, c))
    print("  Partial_ratio:", fuzz.partial_ratio(query, c))
    print("  Token_set_ratio:", fuzz.token_set_ratio(query, c))
    print()


CAND: Ninh Thuận
  WRatio: 81.81818181818181
  Ratio (levenshtein): 81.81818181818181
  Partial_ratio: 94.73684210526316
  Token_set_ratio: 81.81818181818181

CAND: Bắc Ninh
  WRatio: 85.5
  Ratio (levenshtein): 40.0
  Partial_ratio: 66.66666666666667
  Token_set_ratio: 66.66666666666666

CAND: Ninh Thuận Province
  WRatio: 85.5
  Ratio (levenshtein): 58.06451612903225
  Partial_ratio: 85.71428571428572
  Token_set_ratio: 58.064516129032256

CAND: Hiền Ninh
  WRatio: 58.46153846153846
  Ratio (levenshtein): 38.095238095238095
  Partial_ratio: 61.53846153846154
  Token_set_ratio: 61.53846153846154



In [ ]:
from rapidfuzz import process, fuzz

provinces = [
    "Hà Nội", "Hồ Chí Minh", "Đà Nẵng", "Hải Phòng", "Cần Thơ",
    "An Giang", "Bà Rịa - Vũng Tàu", "Bắc Giang", "Bắc Kạn",
    "Bạc Liêu", "Bắc Ninh", "Bến Tre", "Bình Dương", "Bình Định",
    "Bình Phước", "Bình Thuận", "Cà Mau", "Cao Bằng", "Đắk Lắk",
    "Đắk Nông", "Điện Biên", "Đồng Nai", "Đồng Tháp", "Gia Lai",
    "Hà Giang", "Hà Nam", "Hà Tĩnh", "Hải Dương", "Hậu Giang",
    "Hòa Bình", "Hưng Yên", "Khánh Hòa", "Kiên Giang", "Kon Tum",
    "Lai Châu", "Lâm Đồng", "Lạng Sơn", "Lào Cai", "Long An",
    "Nam Định", "Nghệ An", "Ninh Bình", "Ninh Thuận", "Phú Thọ",
    "Phú Yên", "Quảng Bình", "Quảng Nam", "Quảng Ngãi",
    "Quảng Ninh", "Quảng Trị", "Sóc Trăng", "Sơn La", "Tây Ninh",
    "Thái Bình", "Thái Nguyên", "Thanh Hóa", "Thừa Thiên Huế",
    "Tiền Giang", "Trà Vinh", "Tuyên Quang", "Vĩnh Long",
    "Vĩnh Phúc", "Yên Bái"
]

def correct_location(asr_text, provinces, threshold=40):
    match, score, _ = process.extractOne(
        asr_text, provinces, scorer=fuzz.WRatio
    )
    if score >= threshold:
        return match, score
    return None, score

asr_outputs = ["ho chi minh", "ha noi", "ba ria vung tau", "da nangg", "ca mauu"]
asr_outputs = [
    "Bưu gửi được chuyển đến: Ms. Huệ –, Công ty TNHH TM Ngọc Quê. Địa chỉ: Km82 Quốc lộ 1a, thôn Rừng Dông, xã Hữu Lũng, tỉnh Ninh Thuậtttt. Điện thoại liên hệ: 0985 378168.",
    "ninh thuật"
]

for out in asr_outputs[:1]:
    match, score = correct_location(out, provinces)
    print(f"ASR: {out} → Match: {match}, Score: {score}")


ASR: Bưu gửi được chuyển đến: Ms. Huệ –, Công ty TNHH TM Ngọc Quê. Địa chỉ: Km82 Quốc lộ 1a, thôn Rừng Dông, xã Hữu Lũng, tỉnh Ninh Thuậtttt. Điện thoại liên hệ: 0985 378168. → Match: Bắc Ninh, Score: 57.0


In [15]:
from rapidfuzz import process, fuzz

provinces = [
    "Hà Nội", "Hồ Chí Minh", "Đà Nẵng", "Hải Phòng", "Cần Thơ",
    "Bà Rịa - Vũng Tàu", "Cà Mau", "Bình Dương", "Thanh Hóa", "Nghệ An"
]

def find_locations(asr_text, provinces, threshold=40):
    words = asr_text.split()
    found = []
    # thử các n-gram dài tối đa 4 từ
    for n in range(1, 5):
        for i in range(len(words) - n + 1):
            ngram = " ".join(words[i:i+n])
            match, score, _ = process.extractOne(
                ngram, provinces, scorer=fuzz.WRatio
            )
            if score >= threshold:
                found.append((ngram, match, score))
    return found

# ví dụ ASR transcript
asr_texts = [
    "toi dang o ha noi hom nay",
    "toi muon di da nangg thang sau",
    "nha toi o ba ria vung tau",
    "toi sinh ra o ca mauu",
    "toi chuan bi vao thanh pho ho chi minh"
]

for txt in asr_texts:
    print("ASR:", txt)
    matches = find_locations(txt, provinces)
    for ng, match, score in matches:
        print(f"  N-gram: {ng} → Province: {match}, Score: {score}")
    print()


ASR: toi dang o ha noi hom nay
  N-gram: toi → Province: Hà Nội, Score: 45.0
  N-gram: dang → Province: Đà Nẵng, Score: 60.00000000000001
  N-gram: ha → Province: Thanh Hóa, Score: 90.0
  N-gram: noi → Province: Hà Nội, Score: 45.0
  N-gram: hom → Province: Hồ Chí Minh, Score: 45.0
  N-gram: nay → Province: Thanh Hóa, Score: 45.0
  N-gram: toi dang → Province: Hải Phòng, Score: 47.05882352941176
  N-gram: dang o → Province: Hải Phòng, Score: 45.0
  N-gram: o ha → Province: Thanh Hóa, Score: 51.42857142857142
  N-gram: ha noi → Province: Thanh Hóa, Score: 54.0
  N-gram: noi hom → Province: Cần Thơ, Score: 42.85714285714286
  N-gram: hom nay → Province: Nghệ An, Score: 42.85714285714286
  N-gram: toi dang o → Province: Hải Phòng, Score: 42.10526315789473
  N-gram: dang o ha → Province: Thanh Hóa, Score: 44.44444444444444
  N-gram: o ha noi → Province: Hồ Chí Minh, Score: 42.10526315789473
  N-gram: ha noi hom → Province: Thanh Hóa, Score: 42.10526315789473
  N-gram: noi hom nay → Provinc

In [11]:
from rapidfuzz import process

dictionary = ["xây", "dựng", "đầu", "tư", "cổ", "phần", "Liên", "Việt", "Postbank", "Bình", "Minh"]

def fuzzy_correct(word, dictionary, threshold=65):
    match = process.extractOne(word, dictionary, score_cutoff=threshold)
    if match:
        return match[0]  # từ gần nhất
    return word  # giữ nguyên nếu không tìm thấy

text = "công ty cổ phần đầu tư và xay dựng bành minh"
corrected_words = [fuzzy_correct(w, dictionary) for w in text.split()]
print(" ".join(corrected_words))
# -> "công ty cổ phần đầu tư và xây dựng Bình Minh"


công ty cổ phần đầu tư và xây dựng bành Minh


In [2]:
from rapidfuzz import fuzz, process

word = "xay"
dictionary = ["xây", "dựng", "đầu tư", "công ty"]

# So sánh trực tiếp
print(fuzz.ratio(word, "xây"))   # ~67
print(fuzz.ratio(word, "dựng")) # ~0

# Tìm từ gần nhất
best_match = process.extractOne(word, dictionary)
print(best_match)  
# -> ('xây', 67.0, 0) (kết quả, score, index)


66.66666666666667
0.0
('xây', 66.66666666666667, 0)


In [12]:
from rapidfuzz import process

dictionary_phrases = [
    "Liên Việt Postbank", 
    "Bình Minh", 
    "xây dựng", 
    "cổ phần đầu tư"
]

def fuzzy_correct_phrase(text, dictionary, threshold=70):
    match = process.extractOne(text, dictionary, score_cutoff=threshold)
    if match:
        return match[0]
    return text

text = "công ty cổ phần đầu tư và xay dựng bành minh"
corrected = fuzzy_correct_phrase(text, dictionary_phrases)
print(corrected)
# -> "công ty cổ phần đầu tư và xây dựng Bình Minh"


cổ phần đầu tư


In [13]:
from rapidfuzz import process

# Bộ từ điển chuẩn
dictionary = ["xây", "dựng", "đầu", "tư", "cổ", "phần",
              "Liên", "Việt", "Postbank", "Bình", "Minh"]

# Dữ liệu test (giả sử là output từ ASR + ground truth)
test_cases = [
    ("công ty cổ phần đầu tư và xay dựng bành minh", 
     "công ty cổ phần đầu tư và xây dựng Bình Minh"),
    ("liên việt vốt spanh", 
     "Liên Việt Postbank"),
    ("dau tu co phan binh minh", 
     "đầu tư cổ phần Bình Minh"),
]

def fuzzy_correct(word, dictionary, threshold=80):
    match = process.extractOne(word, dictionary, score_cutoff=threshold)
    if match:
        return match[0]
    return word

def correct_text(text, dictionary, threshold=80):
    return " ".join(fuzzy_correct(w, dictionary, threshold) for w in text.split())

# Thử với nhiều ngưỡng
thresholds = [60, 70, 80]

for thr in thresholds:
    print(f"\n=== Threshold {thr} ===")
    for asr, gold in test_cases:
        corrected = correct_text(asr, dictionary, threshold=thr)
        print(f"ASR:       {asr}")
        print(f"Corrected: {corrected}")
        print(f"Expected:  {gold}")
        print("-" * 40)




=== Threshold 60 ===
ASR:       công ty cổ phần đầu tư và xay dựng bành minh
Corrected: cổ xây cổ phần đầu tư và xây dựng bành Minh
Expected:  công ty cổ phần đầu tư và xây dựng Bình Minh
----------------------------------------
ASR:       liên việt vốt spanh
Corrected: Liên Việt tư spanh
Expected:  Liên Việt Postbank
----------------------------------------
ASR:       dau tu co phan binh minh
Corrected: dau đầu co phần Minh Minh
Expected:  đầu tư cổ phần Bình Minh
----------------------------------------

=== Threshold 70 ===
ASR:       công ty cổ phần đầu tư và xay dựng bành minh
Corrected: công ty cổ phần đầu tư và xay dựng bành Minh
Expected:  công ty cổ phần đầu tư và xây dựng Bình Minh
----------------------------------------
ASR:       liên việt vốt spanh
Corrected: Liên Việt vốt spanh
Expected:  Liên Việt Postbank
----------------------------------------
ASR:       dau tu co phan binh minh
Corrected: dau tu co phần Minh Minh
Expected:  đầu tư cổ phần Bình Minh
----------------